# Demo 14 · The way back, timed

Today you write the smoke test that can say a deployment is bad. This demo is about the
minute after it says so.

Two teams ship the same bad release at 16:40. Both have a rollback plan. One plan is a
sentence in a README nobody has run. The other is a command somebody ran on Monday. Same
incident, same probe, two very different evenings.

About five minutes. No key, no network: the deployments are Python functions, and the
clock is a counter, so it runs in the same second on every machine.

In [ ]:
# Two releases of the same service. v2 stopped validating: it answers garbage with 200.
GOOD_ANSWER = {"answer": "Chunking splits a document into passages.", "citations": ["rag-basics"]}


def release_v1(path, body=None):
    if isinstance(body, dict) and isinstance(body.get("question"), str):
        return {"status": 200, "body": GOOD_ANSWER}
    return {"status": 400, "body": {"error": "'question' must be a non-empty string"}}


def release_v2(path, body=None):
    if isinstance(body, dict) and isinstance(body.get("question"), str):
        return {"status": 200, "body": GOOD_ANSWER}
    return {"status": 200, "body": {"answer": "Yes, that is correct.", "citations": []}}


RELEASES = {"v1": release_v1, "v2": release_v2}


class Deployment:
    """What is live, and a clock that only moves when somebody does something."""

    def __init__(self):
        self.live = "v1"
        self.previous = None
        self.clock_s = 0

    def ship(self, version):
        self.previous, self.live = self.live, version
        self.clock_s += 30

    def serve(self, path, body=None):
        return RELEASES[self.live](path, body)


def broken(deployment):
    """One probe: the malformed body. A 200 here means the boundary stopped validating."""
    return deployment.serve("/answer", {"quesiton": 12})["status"] == 200


print("the probe, against each release:", {v: RELEASES[v]("/answer", {"quesiton": 12})["status"] for v in RELEASES})

## 1. Two plans for the same incident

Read them as their authors wrote them. Both sound reasonable on a Monday.

In [ ]:
PLAN_A = "If something breaks, roll back to the previous version."


def plan_a(deployment):
    # Never run before tonight. 'Roll back' to what? Somebody goes looking: the
    # previous image tag, who has access, whether the database moved. Each question
    # is minutes, and the clock is running.
    for question in ("which version was previous?", "who can deploy?", "did the schema change?"):
        deployment.clock_s += 600
    return False  # still guessing at 17:10


PLAN_B = "If the malformed probe returns 200, ship the previous version (`ship(previous)`): under 60 s."


def plan_b(deployment):
    # Rehearsed on Monday: the command, the version it goes back to, and how long it takes.
    deployment.ship(deployment.previous)
    return True


print("plan A:", PLAN_A)
print("plan B:", PLAN_B)

## 2. 16:40: v2 ships, the probe fires, each team runs its plan

In [ ]:
for name, plan in (("A, the sentence", plan_a), ("B, the command", plan_b)):
    deployment = Deployment()
    deployment.ship("v2")
    shipped_at = deployment.clock_s
    detected = broken(deployment)
    acted = plan(deployment) if detected else False
    recovered = acted and not broken(deployment)
    minutes = (deployment.clock_s - shipped_at) / 60
    state = f"back on {deployment.live}" if recovered else f"still on {deployment.live}"
    print(f"plan {name:<16} detected: {detected}  {state:<14} after {minutes:.1f} min")

Same probe, same incident. Plan A spent half an hour on questions somebody could have
answered on Monday, and v2 is still answering garbage with 200. Plan B was one command
with a number in it, and it was back in under a minute.

**A rollback you have not run is a hope.** The `rollback` field in today's report asks
for an action, a number and a unit for exactly this reason. The smoke test that decides
*when* to run it is today's lab, `ch14-e3`, and that part is yours:
[the smoke test as a contract](../units/en/unit3/session-14-deploy-and-operate/concepts-3.mdx).

## 3. The same idea in your capstone

Your Gecko capstone already has plan B, and it is tested by a command:

```bash
make smoke             # the five cases and the trap, live on devnet
make smoke-recorded    # the rollback: the same smoke, on recorded answers
```

`GECKO_SOURCE=recorded` is the switch. `make smoke-recorded` is the rehearsal. On Friday,
if devnet or Gecko is down at minute 1:30, you run the switch and say so.

## Your turn

Never marked, never submitted.

1. Rewrite plan A so it calls `deployment.ship(deployment.previous)` instead of asking its
   three questions. It recovers now; what is still missing from its sentence that plan B
   has?
2. Make v2 fail differently: answer the malformed body with 503. Does the one probe still
   catch it? What would your four-field report say?
3. Write your own capstone's rollback in the shape of plan B: the trigger, the command,
   and how long it takes. It goes into your `docs/DEFENCE.md`.

**What to take away:** write the way back while nothing is broken, and run it once
before you need it.